# Investigation 3 — Land-cover change and design runoff

**Participant-directed investigation.** Quantify a mapped curve-number
trajectory while holding the watershed boundary, soil layer, lookup,
hydrologic condition, scale, and aggregation convention fixed. Then
compare the temporal signal with a hydrologic-condition sensitivity
interval and its effect on design runoff.

**Minimum result:** a CN trajectory, start-to-end change, poor-to-good
condition spread, and a statement identifying which inputs changed and
which were controlled.


In [ ]:
# V3 portable setup: local repository, GitHub Pages bundle, or Colab.
from pathlib import Path
import hashlib
import importlib
import importlib.util
import os
import subprocess
import sys
import urllib.request
import zipfile

CNKIT_VERSION = "1.1.0"
BUNDLE_URL = (
    "https://skp703.github.io/cn-workshop-2026/"
    "downloads/cn_workshop_v3_data.zip"
)
BUNDLE_SHA256 = "925861246fe9520c4b7f399227ca6133e60f27a5063a73c9fb6715cd9904780c"


def _is_workshop_root(path):
    return (path / "data" / "sites.csv").exists() and (path / "prepared").exists()


def _find_workshop_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content/cnkit_workshop"),
    ]
    requested = os.environ.get("CNKIT_WORKSHOP_HOME")
    if requested:
        candidates.insert(0, Path(requested).expanduser())
    for candidate in candidates:
        candidate = candidate.resolve()
        if _is_workshop_root(candidate):
            return candidate, "existing workshop folder"

    destination = Path("/content/cnkit_workshop") if Path("/content").exists() else Path.cwd() / ".cnkit_workshop"
    destination.mkdir(parents=True, exist_ok=True)
    archive = destination / "cn_workshop_v3_data.zip"
    print("Downloading the versioned V3 workshop bundle...")
    request = urllib.request.Request(BUNDLE_URL, headers={"User-Agent": "cn-workshop-v3"})
    with urllib.request.urlopen(request, timeout=120) as response, archive.open("wb") as handle:
        handle.write(response.read())
    digest = hashlib.sha256(archive.read_bytes()).hexdigest()
    if digest != BUNDLE_SHA256:
        raise RuntimeError(
            "Workshop bundle checksum mismatch. Expected %s, received %s. "
            "Delete %s and try again." % (BUNDLE_SHA256, digest, archive)
        )
    with zipfile.ZipFile(archive) as zipped:
        zipped.extractall(destination)
    if not _is_workshop_root(destination):
        raise RuntimeError("The workshop bundle downloaded but required files are missing.")
    return destination.resolve(), "checksum-verified workshop download"


WORKSHOP_ROOT, DATA_SOURCE = _find_workshop_root()
DATA_DIR = WORKSHOP_ROOT / "data"
PREPARED_DIR = WORKSHOP_ROOT / "prepared"


def _load_cnkit():
    try:
        import cnkit as package
        if getattr(package, "__version__", None) == CNKIT_VERSION:
            return package, "installed package"
    except ImportError:
        pass

    for candidate in [
        WORKSHOP_ROOT / "vendor" / "cnkit.py",
        WORKSHOP_ROOT / "cnkit.py",
        Path.cwd() / "vendor" / "cnkit.py",
        Path.cwd().parent / "vendor" / "cnkit.py",
    ]:
        if candidate.exists():
            spec = importlib.util.spec_from_file_location("cnkit", candidate)
            package = importlib.util.module_from_spec(spec)
            sys.modules["cnkit"] = package
            spec.loader.exec_module(package)
            return package, str(candidate)

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "cnkit==" + CNKIT_VERSION]
    )
    importlib.invalidate_caches()
    import cnkit as package
    return package, "PyPI"


def activate_full_cnkit():
    """Return the installed package with data, delineation, and GEE modules."""
    global cnkit, CNKIT_SOURCE
    if hasattr(cnkit, "__path__"):
        return cnkit
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "cnkit[gee]==" + CNKIT_VERSION]
    )
    for name in [key for key in sys.modules if key == "cnkit" or key.startswith("cnkit.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
    cnkit = importlib.import_module("cnkit")
    CNKIT_SOURCE = "PyPI with Earth Engine dependencies"
    return cnkit


cnkit, CNKIT_SOURCE = _load_cnkit()
print("cnkit version:", getattr(cnkit, "__version__", CNKIT_VERSION + " workshop module"))
print("cnkit source :", CNKIT_SOURCE)
print("data source  :", DATA_SOURCE)
print("data folder  :", DATA_DIR)
print("setup complete")


In [ ]:
import json
import os
from getpass import getpass

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from cnkit import runoff

USE_EARTH_ENGINE = False
DELINEATION_INPUT = "gage"       # "gage" or "outlet"
GAGE = "01646000"
LAT, LON = 38.97594, -77.24581
YEARS = [2001, 2004, 2007, 2010, 2013, 2016, 2019]
SOILS_SOURCE = "sda"
DESIGN_DEPTH_IN = 4.78


## Step 1 — Define what changes and what remains fixed

For year $t$, the area-weighted curve number is

$$
CN_t=\frac{\sum_i A_{i,t}CN_i}{\sum_i A_{i,t}},
$$

where $A_{i,t}$ is the area of land-cover–soil pair $i$ in year
$t$, and $CN_i$ is the lookup value assigned to that pair. Across
the trajectory, `cnkit` holds the watershed boundary, soil layer,
lookup crosswalk, hydrologic condition, pixel scale, and compositing
convention constant. Annual NLCD is the changing input.

Consequently, the year-to-year difference is attributable to mapped
land-cover change under those fixed analytical choices. It is not a
direct measurement of infiltration, storage, or runoff.


### Earth Engine data sources for this trajectory

The table below records the exact Earth Engine asset identifiers used in
the guided exercise and the optional live notebook pathway. An asset
identifier documents the computational input; the agency product remains
the scientific data source that should be cited in a report.

| Workshop use | Publisher and product | Exact Earth Engine asset | Relevant band or object | Native scale | Catalog status |
|---|---|---|---|---:|---|
| Guided watershed geometry | U.S. Geological Survey, Watershed Boundary Dataset, HUC12 | `USGS/WBD/2017/HUC12` | `FeatureCollection`; field `huc12` | vector | Official Earth Engine catalog |
| Guided land cover and imperviousness | U.S. Geological Survey, NLCD 2019 release | `USGS/NLCD_RELEASES/2019_REL/NLCD` | `landcover`; `impervious` | 30 m | Official Earth Engine catalog |
| Live annual land-cover distribution and trajectory | U.S. Geological Survey, Annual NLCD Collection 1 | `projects/sat-io/open-datasets/USGS/ANNUAL_NLCD/LANDCOVER` | first band of each annual image | 30 m | Earth Engine Community Catalog mirror |
| Live mean fractional impervious surface | U.S. Geological Survey, Annual NLCD Collection 1 | `projects/sat-io/open-datasets/USGS/ANNUAL_NLCD/FRACTIONAL_IMPERVIOUS_SURFACE` | first band of each annual image | 30 m | Earth Engine Community Catalog mirror |
| Live soil map-unit keys for `SOILS_SOURCE="sda"` | USDA Natural Resources Conservation Service, gNATSGO | `projects/sat-io/open-datasets/gNATSGO/raster/mukey` | first band; integer map-unit key | 30 m | Earth Engine Community Catalog mirror |
| Optional global HSG comparison for `SOILS_SOURCE="hihydro"` | FutureWater, HiHydroSoil v2.0 | `projects/sat-io/open-datasets/HiHydroSoilv2_0/Hydrologic_Soil_Group_250m` | first band; modeled HSG class | 250 m | Earth Engine Community Catalog asset |

**Supporting services that are not Earth Engine assets.** The USGS NLDI
[web service](https://api.water.usgs.gov/docs/nldi/) constructs the live watershed
boundary before `cnkit` converts it to an Earth Engine geometry. For the
`sda` soil pathway, Earth Engine supplies only the gNATSGO map-unit keys;
the USDA NRCS Soil Data Access
[web service](https://sdmdataaccess.sc.egov.usda.gov/WebServiceHelp.aspx)
supplies the dominant-condition hydrologic-soil-group attributes joined to
those keys.

**Source and access documentation.** See the USGS
[Annual NLCD product page](https://www.usgs.gov/centers/eros/science/annual-national-land-cover-database),
USDA NRCS [gNATSGO documentation](https://www.nrcs.usda.gov/resources/data-and-reports/gridded-national-soil-survey-geographic-database-gnatsgo),
the Earth Engine catalog entries for
[WBD HUC12](https://developers.google.com/earth-engine/datasets/catalog/USGS_WBD_2017_HUC12)
and [NLCD 2019](https://developers.google.com/earth-engine/datasets/catalog/USGS_NLCD_RELEASES_2019_REL_NLCD),
and the [Earth Engine Community Catalog Annual NLCD record](https://gee-community-catalog.org/projects/annual_nlcd/).

The `projects/sat-io` paths are community-hosted access copies. Record the
exact asset ID and retrieval date for reproducibility, but cite the original
agency product as the data source. Availability of annual images is checked
at runtime rather than assumed from a fixed list of years.


## Step 2 — Load the recorded Earth Engine trajectory

The table below was produced by the live workflow for Difficult Run.
It contains poor, fair, and good hydrologic-condition calculations for
the same annual joint land-cover–soil distribution. The three columns
differ only in the lookup-table condition row.


In [ ]:
trajectory = pd.read_csv(PREPARED_DIR / "difficult_run_gee_trajectory.csv").set_index("year")
recorded = json.loads(
    (PREPARED_DIR / "difficult_run_gee_summary.json").read_text()
)
change = float(trajectory.fair.iloc[-1] - trajectory.fair.iloc[0])
mean_spread = float(trajectory.spread.mean())
ratio = mean_spread / abs(change)

print("boundary method                 NLDI split-catchment upstream")
print("boundary area                   %.4f square miles" % recorded["watershed"]["area_sqmi"])
print("land-cover asset                %s" % recorded["land_cover"]["asset"])
print("soil asset                      %s" % recorded["soils"]["asset"])
print("soil area without mapped HSG    %.2f %%" % recorded["soils"]["percent_area_no_hsg"])
print()
print("2001 CN                         %.4f" % trajectory.fair.iloc[0])
print("2019 CN                         %.4f" % trajectory.fair.iloc[-1])
print("mapped land-cover change       %+.4f CN" % change)
print("mean condition spread           %.4f CN" % mean_spread)
print("condition spread / change        %.1f" % ratio)
display(trajectory.round(4))


## Step 3 — Interpret the hydrologic-condition interval

Poor, fair, and good are field descriptions of cover density,
management, residue, grazing, and related surface conditions. Annual
NLCD classifies land cover but does not observe those attributes.
Recalculating all pixels with the poor and good lookup rows therefore
provides a **sensitivity interval** around the fair-condition result.

This interval is not a statistical confidence interval: no probability
distribution has been assigned to hydrologic condition. Its purpose is
to show how strongly an unobserved table choice affects the trajectory.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.5))

axes[0].plot(
    trajectory.index,
    trajectory.fair,
    "o-",
    color="#007f92",
    lw=2.6,
)
axes[0].set_title("Fair-condition trajectory (expanded scale)")

axes[1].fill_between(
    trajectory.index,
    trajectory.good,
    trajectory.poor,
    color="#c85d45",
    alpha=0.22,
    label="poor-to-good sensitivity interval",
)
axes[1].plot(
    trajectory.index,
    trajectory.fair,
    "o-",
    color="#007f92",
    lw=2.6,
    label="fair-condition trajectory",
)
axes[1].set_title("Trajectory in the condition interval")
axes[1].legend(loc="upper left")

for ax in axes:
    ax.set(xlabel="year", ylabel="composite curve number")
    ax.grid(alpha=0.25)
plt.show()


## Step 4 — Understand how `cn_trajectory` performs the calculation

The convenience workflow organizes the same lower-level operations
used in Investigation 2:

1. Accept an existing `Watershed`, or delineate one from coordinates.
2. Create one `Basin` so the boundary and scale remain fixed.
3. For each requested year, call `joint_landcover_soils`; that method
   packs land-cover and soil codes into one integer image, applies one
   Earth Engine frequency histogram, and decodes the observed pairs.
4. Send the same joint table to `composite_from_areas` for poor, fair,
   and good lookup rows. No additional Earth Engine reduction is needed
   for the condition calculations.
5. Optionally transform each composite CN to runoff at a stated design
   depth, then attach assets, scale, boundary, and unmapped-area
   provenance to the returned result.

The workflow is convenient, but the scientific definition of the
trajectory remains the equation in Step 1.


## Step 4A — Delineate a selected watershed

The application is separated into boundary construction and Earth
Engine analysis so each spatial operation can be inspected. Change the
gage or outlet configuration above, then set `USE_EARTH_ENGINE=True`.


In [ ]:
selected_watershed = None

if USE_EARTH_ENGINE:
    activate_full_cnkit()
    from cnkit.delineate import watershed_from_gage, watershed_from_point

    if DELINEATION_INPUT == "gage":
        selected_watershed = watershed_from_gage(GAGE)
    elif DELINEATION_INPUT == "outlet":
        selected_watershed = watershed_from_point(LAT, LON)
    else:
        raise ValueError("DELINEATION_INPUT must be 'gage' or 'outlet'")

    print(selected_watershed)
    print("area: %.3f square miles" % selected_watershed.area_sqmi)
else:
    print("Recorded Difficult Run trajectory selected.")


## Step 4B — Calculate the annual trajectory

Authentication and the annual reductions occur only in this cell. The
preceding boundary can therefore be checked before any raster summary
is requested.


In [ ]:
live_trajectory = None

if USE_EARTH_ENGINE:
    import ee
    from cnkit.gee import initialise
    from cnkit.workflows import cn_trajectory

    project = os.environ.get("CNKIT_EE_PROJECT") or getpass(
        "Earth Engine project ID: "
    )
    ee.Authenticate()
    initialise(project=project)

    live_trajectory = cn_trajectory(
        watershed=selected_watershed,
        project=project,
        years=YEARS,
        condition="fair",
        soils=SOILS_SOURCE,
        design_depth_in=DESIGN_DEPTH_IN,
        progress=lambda done, total, year: print(
            "%d/%d  %d" % (done, total, year)
        ),
    )
    display(live_trajectory)
    print(live_trajectory.summary())
else:
    print("Set USE_EARTH_ENGINE=True to calculate the selected watershed.")


## Step 5 — Translate the trajectory to design runoff

The nonlinear runoff equation determines whether a small change in CN
has a consequential effect at the selected rainfall depth. Calculate
runoff for the first and last fair-condition CN values and for the poor
and good bounds in the final year.


In [ ]:
design_comparison = pd.Series(
    {
        "first-year fair runoff, in": float(runoff(
            DESIGN_DEPTH_IN, trajectory.fair.iloc[0], lam=0.20
        )),
        "last-year fair runoff, in": float(runoff(
            DESIGN_DEPTH_IN, trajectory.fair.iloc[-1], lam=0.20
        )),
        "last-year poor runoff, in": float(runoff(
            DESIGN_DEPTH_IN, trajectory.poor.iloc[-1], lam=0.20
        )),
        "last-year good runoff, in": float(runoff(
            DESIGN_DEPTH_IN, trajectory.good.iloc[-1], lam=0.20
        )),
    }
)
display(design_comparison.round(4))


## Open investigation — Choose a question

1. Select different start and end years and explain whether the inferred
   trend is stable.
2. Compare CN change with runoff change at two rainfall depths.
3. Use a selected watershed through Earth Engine and compare its signal
   with the Difficult Run reference trajectory.
4. Identify which land-cover transitions would need to be examined to
   explain the mapped trajectory.

Record: boundary and years; controlled inputs; CN and runoff changes;
condition interval; extension result; interpretation; and limitation.


## Method audit

`cn_trajectory` reuses one watershed and `Basin`, requests the joint
land-cover–soil distribution for each year, applies the same lookup and
aggregation conventions, and attaches the spatial provenance. The
analyst selects the years, data sources, scale, hydrologic condition,
design rainfall, and interpretation of the resulting differences.

## References and data sources

- U.S. Geological Survey. [Annual National Land Cover Database](https://www.usgs.gov/centers/eros/science/annual-national-land-cover-database).
- USDA Natural Resources Conservation Service.
  [gNATSGO](https://www.nrcs.usda.gov/resources/data-and-reports/gridded-national-soil-survey-geographic-database-gnatsgo)
  and [Soil Data Access](https://sdmdataaccess.sc.egov.usda.gov/WebServiceHelp.aspx).
- NOAA National Weather Service. [Atlas 14 precipitation-frequency estimates](https://www.weather.gov/owp/hdsc).
- USDA NRCS. 2004. [NEH Part 630, Chapter 10](https://directives.nrcs.usda.gov/sites/default/files2/1712930608/7300.pdf).

The complete cross-notebook source ledger is available in
[workshop source ledger](https://github.com/skp703/cn-workshop-2026/blob/main/docs/SOURCES.md).
